# Training Notebook - A2-V4

This notebook satisfies the A2 submission requirements for the current A2-V4 structure.

Kaggle offline note:
1. This notebook now supports offline pretrained loading from A2-V4 only.
2. Put torchvision checkpoints in A2-V4/pretrained or A2-V4/weights.
3. Supported names include:
   - resnet50-11ad3fa6.pth (or resnet50_imagenet1k_v2.pth)
   - mobilenet_v2-7ebf99e0.pth (or mobilenet_v2_imagenet1k_v1.pth)

It will:
1. Implement PCA using NumPy (Part 1)
2. Write compliant classification/model.py and segmentation/model.py in the submission folder
3. Train classification and segmentation models
4. Tune classification thresholds during training
5. Train two separate segmentation variants (A and B) for statistical testing
6. Use fixed classification threshold 0.5 at inference/submission
7. Generate submission.csv

In [ ]:
print("python kernel check check!")

In [ ]:
from __future__ import annotations

import importlib.util
import json
import random
import sys
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from torchvision.transforms import functional as TF

ROOT = Path('.').resolve()
KAGGLE = Path('/kaggle').exists()

def _find_kaggle_dataset_root() -> Path:
    # Prefer explicitly known mount patterns first for speed/stability.
    direct_candidates = [
        Path('/kaggle/input/datasets/sinfilroamigo/mlds-a2/Dataset'),
        Path('/kaggle/input/mlds-a2/Dataset'),
    ]
    for c in direct_candidates:
        if (c / 'train').exists() and (c / 'test').exists():
            return c

    # Generic fallback: look for any Dataset folder containing train+test.
    for p in Path('/kaggle/input').rglob('Dataset'):
        if (p / 'train').exists() and (p / 'test').exists():
            return p

    raise FileNotFoundError(
        'Could not find Dataset/ under /kaggle/input. Please attach the dataset in Kaggle and rerun.'
    )

if KAGGLE:
    DATA_ROOT = _find_kaggle_dataset_root()
    WORK_ROOT = Path('/kaggle/working')
    SUBMISSION_ROOT = WORK_ROOT # / '25942_MAKWANA_JEVINKUMAR_Assignment_2'
else:
    # Use data/Dataset in the repo; download it from Google Drive if missing.
    from data_utils import ensure_dataset
    DATA_ROOT = ensure_dataset(ROOT / 'data')
    SUBMISSION_ROOT = ROOT

CLASS_DIR = SUBMISSION_ROOT / 'classification'
SEG_DIR = SUBMISSION_ROOT / 'segmentation'
CLASS_WEIGHTS = CLASS_DIR / 'weights'
SEG_WEIGHTS = SEG_DIR / 'weights'

CLASS_DIR.mkdir(parents=True, exist_ok=True)
SEG_DIR.mkdir(parents=True, exist_ok=True)
CLASS_WEIGHTS.mkdir(parents=True, exist_ok=True)
SEG_WEIGHTS.mkdir(parents=True, exist_ok=True)

def _dedupe_paths(paths: List[Path]) -> List[Path]:
    out: List[Path] = []
    seen = set()
    for p in paths:
        key = str(p)
        if key in seen:
            continue
        seen.add(key)
        out.append(p)
    return out

def _build_pretrained_dirs() -> List[Path]:
    dirs: List[Path] = [
        ROOT / 'pretrained',
        ROOT / 'weights',
        Path.cwd() / 'pretrained',
        Path.cwd() / 'weights',
    ]

    if KAGGLE:
        dirs.extend([
            Path('/kaggle/working/pretrained'),
            Path('/kaggle/working/weights'),
            Path('/kaggle/working/MLDS/A2/A2-V4/pretrained'),
            Path('/kaggle/working/MLDS/A2/A2-V4/weights'),
            Path('/kaggle/working/MLDS/A2/A2-V3/pretrained'),
            Path('/kaggle/working/MLDS/A2/A2-V3/weights'),
        ])

        # Also include any mounted folder named pretrained under Kaggle input.
        input_root = Path('/kaggle/input')
        if input_root.exists():
            for p in input_root.rglob('pretrained'):
                dirs.append(p)

    return _dedupe_paths(dirs)

def _mkdir_if_writable(path: Path) -> None:
    # Avoid attempting writes under /kaggle/input (read-only).
    text = str(path)
    if KAGGLE and text.startswith('/kaggle/input'):
        return
    try:
        path.mkdir(parents=True, exist_ok=True)
    except Exception:
        pass

LOCAL_PRETRAINED_DIRS = _build_pretrained_dirs()
for _d in LOCAL_PRETRAINED_DIRS:
    _mkdir_if_writable(_d)

SEARCH_ROOTS = [ROOT, Path.cwd()]
if KAGGLE:
    SEARCH_ROOTS.extend([Path('/kaggle/working'), Path('/kaggle/input')])
SEARCH_ROOTS = _dedupe_paths(SEARCH_ROOTS)

def _extract_state_dict(state):
    if isinstance(state, dict):
        for key in ('state_dict', 'model_state', 'model', 'weights'):
            if key in state and isinstance(state[key], dict):
                return state[key]
    return state

def _strip_module_prefix_if_needed(state_dict: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    if not state_dict:
        return state_dict
    keys = list(state_dict.keys())
    if all(k.startswith('module.') for k in keys):
        return {k[len('module.'):]: v for k, v in state_dict.items()}
    return state_dict

def _try_load_weights_from_path(model: nn.Module, ckpt_path: Path, tag: str, strict: bool) -> bool:
    try:
        raw_state = torch.load(ckpt_path, map_location='cpu')
        state = _extract_state_dict(raw_state)
        if not isinstance(state, dict):
            return False
        state = _strip_module_prefix_if_needed(state)
        model.load_state_dict(state, strict=strict)
        print(f'[{tag}] Loaded local pretrained weights from: {ckpt_path}', flush=True)
        return True
    except Exception as e:
        print(f'[{tag}] Found {ckpt_path} but failed to load ({e}). Trying next candidate.', flush=True)
        return False

def _load_local_torchvision_weights(
    model: nn.Module,
    candidate_filenames: List[str],
    tag: str,
    fallback_error: Exception,
    strict: bool = False,
    ) -> nn.Module:
    # Pass 1: direct lookup in known directories.
    for base_dir in LOCAL_PRETRAINED_DIRS:
        for name in candidate_filenames:
            ckpt_path = base_dir / name
            if ckpt_path.exists() and _try_load_weights_from_path(model, ckpt_path, tag, strict):
                return model

    # Pass 2: recursive search in major roots for exact file names.
    print(f'[{tag}] Direct lookup failed; trying recursive search for: {candidate_filenames}', flush=True)
    seen = set()
    for name in candidate_filenames:
        for root in SEARCH_ROOTS:
            if not root.exists():
                continue
            try:
                for found in root.rglob(name):
                    key = str(found)
                    if key in seen:
                        continue
                    seen.add(key)
                    if _try_load_weights_from_path(model, found, tag, strict):
                        return model
            except Exception:
                continue

    print(f'[{tag}] Could not use torchvision download ({fallback_error}); using random init.', flush=True)
    print(f'[{tag}] Checked dirs: {LOCAL_PRETRAINED_DIRS}', flush=True)
    return model

_ORIG_RESNET50 = models.resnet50
_ORIG_MOBILENET_V2 = models.mobilenet_v2

def _resnet50_offline_safe(*args, **kwargs):
    requested_weights = kwargs.get('weights', None)
    if requested_weights is None:
        return _ORIG_RESNET50(*args, **kwargs)
    try:
        return _ORIG_RESNET50(*args, **kwargs)
    except Exception as e:
        base_kwargs = dict(kwargs)
        base_kwargs['weights'] = None
        model = _ORIG_RESNET50(*args, **base_kwargs)
        candidates = [
            'resnet50-11ad3fa6.pth',
            'resnet50_imagenet1k_v2.pth',
            'resnet50_imagenet.pth',
            'resnet50.pth',
        ]
        return _load_local_torchvision_weights(model, candidates, 'CLS', e, strict=False)

def _mobilenet_v2_offline_safe(*args, **kwargs):
    requested_weights = kwargs.get('weights', None)
    if requested_weights is None:
        return _ORIG_MOBILENET_V2(*args, **kwargs)
    try:
        return _ORIG_MOBILENET_V2(*args, **kwargs)
    except Exception as e:
        base_kwargs = dict(kwargs)
        base_kwargs['weights'] = None
        model = _ORIG_MOBILENET_V2(*args, **base_kwargs)
        candidates = [
            'mobilenet_v2-7ebf99e0.pth',
            'mobilenet_v2_imagenet1k_v1.pth',
            'mobilenet_v2_imagenet.pth',
            'mobilenet_v2.pth',
        ]
        return _load_local_torchvision_weights(model, candidates, 'SEG', e, strict=False)

models.resnet50 = _resnet50_offline_safe
models.mobilenet_v2 = _mobilenet_v2_offline_safe

CLASSES:List[str] = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor',
]
NUM_CLASSES = 21

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('ROOT:', ROOT)
print('CWD:', Path.cwd())
print('KAGGLE mode:', KAGGLE)
print('DATA_ROOT (read):', DATA_ROOT)
print('SUBMISSION_ROOT (write):', SUBMISSION_ROOT)
print('Offline pretrained dirs:', LOCAL_PRETRAINED_DIRS)
print('Search roots:', SEARCH_ROOTS)
print('Using device:', device)

## 0) Part 1 - PCA Using NumPy

This section implements PCA from scratch with NumPy on resized grayscale train images.
It reports explained variance and reconstruction error for different component counts.

Assignemnt instruction for PCA:

"Part 1: PCA using NumPy (3 Marks) In the first part of the assignment, participants must implement Principal Component Analysis (PCA) using the NumPy library(If the data is too big to work with resize the images to 32x32 or 16x16 using skimage library). The dataset provided contains high-dimensional data, and the goal is to reduce the dimensionality using PCA while preserving the most important information. Participants will be evaluated based on their implementation, experimentation with different components. Submit Results and observations in code file itself as 1st assignment"

In [ ]:
train_dir = DATA_ROOT / "train"
images_dir = train_dir / "images"
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

image_paths = sorted(
    p for p in images_dir.rglob("*")
    if p.is_file() and p.suffix.lower() in image_extensions
)

print(f"Number of images: {len(image_paths)}")
print("\nImage details (W × H, file size):")

for path in image_paths:
    with Image.open(path) as img:
        width, height = img.size

    size_bytes = path.stat().st_size
    size_text = f"{size_bytes / 1024:.2f} KB"
    if size_bytes >= 1024 * 1024:
        size_text = f"{size_bytes / (1024 * 1024):.2f} MB"

    print(f"{path.name}: {width} × {height}, {size_text}")

total_bytes = sum(p.stat().st_size for p in train_dir.rglob("*") if p.is_file())
print(f"\nTotal train dataset size: {total_bytes / (1024 * 1024):.2f} MB")

In [ ]:
def _min_image_dimension(path):
    with Image.open(path) as image:
        return min(image.size)  # min(width, height)

max_possible_k = min(_min_image_dimension(path) for path in image_paths)
max_possible_k

In [ ]:
import matplotlib.pyplot as plt

PCA_IMAGE_SIZE = 32
PCA_NUM_IMAGES = 300
PCA_COMPONENT_CANDIDATES = [8, 16, 32, 64, 128]

def load_pca_matrix(image_dir: Path, n_images: int, image_size: int) -> np.ndarray:
    paths = sorted(image_dir.glob('*.jpg'))
    if len(paths) < n_images:
        paths = sorted(image_dir.glob('*.png'))
    if not paths:
        raise FileNotFoundError(f'No train images found in {image_dir}')

    use_paths = paths[:n_images]
    rows = []
    for p in use_paths:
        img = Image.open(p).convert('L')
        img = img.resize((image_size, image_size), Image.BILINEAR)
        arr = np.asarray(img, dtype=np.float32) / 255.0
        rows.append(arr.reshape(-1))
    return np.stack(rows, axis=0)

def fit_pca_numpy(X: np.ndarray):
    mean_vec = X.mean(axis=0, keepdims=True)
    Xc = X - mean_vec
    # SVD-based PCA: principal axes are rows of Vt.
    _, s, vt = np.linalg.svd(Xc, full_matrices=False)
    eigvals = (s ** 2) / max(1, (Xc.shape[0] - 1))
    explained_ratio = eigvals / np.sum(eigvals)
    return mean_vec, vt, explained_ratio

def pca_reconstruct(X: np.ndarray, mean_vec: np.ndarray, components: np.ndarray, k: int) -> np.ndarray:
    basis = components[:k]
    Z = (X - mean_vec) @ basis.T
    Xhat = Z @ basis + mean_vec
    return Xhat

X_pca = load_pca_matrix(DATA_ROOT / 'train' / 'images', PCA_NUM_IMAGES, PCA_IMAGE_SIZE)
mean_vec, components, explained_ratio = fit_pca_numpy(X_pca)
cum_explained = np.cumsum(explained_ratio)

valid_ks = [k for k in PCA_COMPONENT_CANDIDATES if k <= components.shape[0]]
recon_rows = []
for k in valid_ks:
    Xhat = pca_reconstruct(X_pca, mean_vec, components, k)
    mse = float(np.mean((X_pca - Xhat) ** 2))
    recon_rows.append({
        'k_components': k, 
        'explained_variance_%': float(cum_explained[k - 1] * 100.0),
        'reconstruction_mse': mse,
    })

pca_report = pd.DataFrame(recon_rows)
print('PCA input matrix shape:', X_pca.shape)
display(pca_report)

plt.figure(figsize=(6, 4))
plt.plot(np.arange(1, len(cum_explained) + 1), cum_explained, linewidth=2)
plt.xlabel('Number of components (k)')
plt.ylabel('Cumulative explained variance')
plt.title('PCA explained variance curve')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

k_show = valid_ks[min(2, len(valid_ks) - 1)] if valid_ks else 8
Xhat_show = pca_reconstruct(X_pca, mean_vec, components, k_show)

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for i in range(4):
    axes[0, i].imshow(X_pca[i].reshape(PCA_IMAGE_SIZE, PCA_IMAGE_SIZE), cmap='gray')
    axes[0, i].axis('off')
    axes[0, i].set_title(f'Original #{i+1}')

    axes[1, i].imshow(Xhat_show[i].reshape(PCA_IMAGE_SIZE, PCA_IMAGE_SIZE), cmap='gray')
    axes[1, i].axis('off')
    axes[1, i].set_title(f'Recon (k={k_show})')

plt.tight_layout()
plt.show()

best_row = pca_report.sort_values('reconstruction_mse').iloc[0]
print(f"Best tested k by reconstruction MSE: {int(best_row['k_components'])}")
print(f"Explained variance at best tested k: {best_row['explained_variance_%']:.2f}%")

### PCA Observations

1. Cumulative explained variance increases monotonically with number of components k.
2. Reconstruction MSE decreases as k increases.
3. Choose k by balancing compression (smaller k) and fidelity (lower MSE).
4. For assignment reporting, cite the k that gives a good variance-MSE tradeoff from the table above.

## 1) Write Compliant `model.py` Files

In [ ]:
classification_model_code = '''
from __future__ import annotations

from pathlib import Path
from typing import Dict, List

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torchvision import models, transforms

CLASSES: List[str] = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor',
]

class ClassifierNet(nn.Module):
    def __init__(self, num_classes: int = 20):
        super().__init__()
        backbone = models.resnet50(weights=None)
        self.backbone = nn.Sequential(*list(backbone.children())[:-2])
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.head = nn.Sequential(
            nn.Linear(2048, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.35),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.25),
            nn.Linear(256, num_classes),

    def __init__(self, weights_dir: str):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = ClassifierNet(num_classes=len(CLASSES)).to(self.device)
        weights_path = Path(weights_dir) / 'best_classifier.pth'
        if not weights_path.exists():
            raise FileNotFoundError(f'Classifier weights not found: {weights_path}')
        state = torch.load(weights_path, map_location='cpu')
        if isinstance(state, dict):
            state = state.get('state_dict', state.get('model_state', state))
        self.model.load_state_dict(state, strict=True)
        self.model.eval()
        self.transform = transforms.Compose([
            transforms.Resize(336),
            transforms.CenterCrop(320),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def predict(self, image: np.ndarray) -> Dict[str, float]:
        if image.dtype != np.uint8:
            image = image.astype(np.uint8)
        pil_img = Image.fromarray(image, mode='RGB')
        x = self.transform(pil_img).unsqueeze(0).to(self.device)
        with torch.no_grad():
            probs = torch.sigmoid(self.model(x)).squeeze(0).detach().cpu().numpy()
        return {name: float(prob) for name, prob in zip(CLASSES, probs)}
'''

segmentation_model_code = '''
from __future__ import annotations

from pathlib import Path
from typing import List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torchvision import models, transforms

CLASSES: List[str] = [
    'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow',
    'diningtable', 'dog', 'horse', 'motorbike', 'person',
    'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor',
]
NUM_CLASSES = 21

class ConvBNAct(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int = 3, stride: int = 1, padding: int | None = None):
        super().__init__()
        if padding is None:
            padding = kernel_size // 2
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride, padding=padding, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class SegmentationBackbone(nn.Module):
    def __init__(self, pretrained_backbone: bool = True):
        super().__init__()
        weights = models.ResNet50_Weights.IMAGENET1K_V2 if pretrained_backbone else None
        backbone = models.resnet50(weights=weights)
        self.stem = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
        )
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4

    def forward(self, x: torch.Tensor):
        x = self.stem(x)
        c1 = self.layer1(x)
        c2 = self.layer2(c1)
        c3 = self.layer3(c2)
        c4 = self.layer4(c3)
        return c1, c2, c3, c4


class SegmentationDecoder(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.proj4 = ConvBNAct(2048, 256, kernel_size=1, padding=0)
        self.fuse34 = nn.Sequential(
            ConvBNAct(256 + 1024, 256),
            ConvBNAct(256, 256),
        )
        self.fuse23 = nn.Sequential(
            ConvBNAct(256 + 512, 192),
            ConvBNAct(192, 192),
        )
        self.fuse12 = nn.Sequential(
            ConvBNAct(192 + 256, 128),
            ConvBNAct(128, 128),
        )
        self.classifier = nn.Conv2d(128, num_classes, kernel_size=1)

    def forward(self, c1: torch.Tensor, c2: torch.Tensor, c3: torch.Tensor, c4: torch.Tensor, out_size: Tuple[int, int]) -> torch.Tensor:
        x = self.proj4(c4)
        x = F.interpolate(x, size=c3.shape[-2:], mode='bilinear', align_corners=False)
        x = self.fuse34(torch.cat([x, c3], dim=1))
        x = F.interpolate(x, size=c2.shape[-2:], mode='bilinear', align_corners=False)
        x = self.fuse23(torch.cat([x, c2], dim=1))
        x = F.interpolate(x, size=c1.shape[-2:], mode='bilinear', align_corners=False)
        x = self.fuse12(torch.cat([x, c1], dim=1))
        x = F.interpolate(x, size=out_size, mode='bilinear', align_corners=False)
        return self.classifier(x)


class CustomSegmentationNet(nn.Module):
    def __init__(self, num_classes: int = NUM_CLASSES, pretrained_backbone: bool = True):
        super().__init__()
        self.backbone = SegmentationBackbone(pretrained_backbone=pretrained_backbone)
        self.decoder = SegmentationDecoder(num_classes=num_classes)

    def forward(self, x: torch.Tensor):
        c1, c2, c3, c4 = self.backbone(x)
        logits = self.decoder(c1, c2, c3, c4, x.shape[-2:])
        return {'out': logits}


class SegmentationModel:
    def __init__(self, weights_dir: str):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = CustomSegmentationNet(num_classes=NUM_CLASSES).to(self.device)
        ckpt_path = Path(weights_dir) / 'best_segmentation.pth'
        if not ckpt_path.exists():
            raise FileNotFoundError(f'Segmentation weights not found: {ckpt_path}')
        state = torch.load(ckpt_path, map_location='cpu')
        if isinstance(state, dict) and 'model_state' in state:
            state = state['model_state']
        self.model.load_state_dict(state, strict=True)
        self.model.eval()
        self.transform = transforms.Compose([
            transforms.Resize((448, 448)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def predict(self, image: np.ndarray) -> np.ndarray:
        if image.dtype != np.uint8:
            image = image.astype(np.uint8)
        h, w = image.shape[:2]
        pil_img = Image.fromarray(image, mode='RGB')
        x = self.transform(pil_img).unsqueeze(0).to(self.device)
        with torch.no_grad():
            logits = self.model(x)['out']
            logits_flip = torch.flip(self.model(torch.flip(x, dims=[3]))['out'], dims=[3])
            pred = torch.argmax(0.5 * (logits + logits_flip), dim=1).squeeze(0).cpu().numpy().astype(np.uint8)
        if pred.shape != (h, w):
            pred = np.array(Image.fromarray(pred).resize((w, h), Image.NEAREST), dtype=np.uint8, copy=True)
        pred[pred == 255] = 0
        return pred
'''

(CLASS_DIR / 'model.py').write_text(classification_model_code, encoding='utf-8')
(SEG_DIR / 'model.py').write_text(segmentation_model_code, encoding='utf-8')
print('Wrote model.py files:')
print(' -', CLASS_DIR / 'model.py')
print(' -', SEG_DIR / 'model.py')


In [ ]:
# Quick sanity checks for local from-scratch workflow
required_dirs = [CLASS_DIR, SEG_DIR, CLASS_WEIGHTS, SEG_WEIGHTS, DATA_ROOT / 'train', DATA_ROOT / 'test']
for p in required_dirs:
    if not p.exists():
        raise FileNotFoundError(f'Missing required path: {p}')
print('A2-V4 local structure check passed.')

## 2) Train Classification + Tune Thresholds

In [ ]:
class ClassificationTrainDataset(Dataset):
    def __init__(self, df: pd.DataFrame, img_dir: Path, train: bool, image_size: int = 320):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.train = train
        self.image_size = image_size

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        img = Image.open(self.img_dir / f"{row['image_id']}.jpg").convert('RGB')

        if self.train:
            if random.random() < 0.5:
                img = TF.hflip(img)
            if random.random() < 0.25:
                img = TF.vflip(img)
            img = TF.resize(img, [336, 336], interpolation=TF.InterpolationMode.BILINEAR)
            i, j, h, w = transforms.RandomResizedCrop.get_params(img, scale=(0.60, 1.0), ratio=(0.80, 1.20))
            img = TF.resized_crop(img, i, j, h, w, [self.image_size, self.image_size], TF.InterpolationMode.BILINEAR)
            if random.random() < 0.35:
                img = TF.adjust_brightness(img, 0.75 + 0.5 * random.random())
            if random.random() < 0.35:
                img = TF.adjust_contrast(img, 0.75 + 0.5 * random.random())
            if random.random() < 0.25:
                img = TF.adjust_saturation(img, 0.75 + 0.5 * random.random())
            if random.random() < 0.15:
                img = TF.adjust_hue(img, -0.10 + 0.20 * random.random())
            if random.random() < 0.25:
                img = TF.rotate(img, -15 + 30 * random.random(), interpolation=TF.InterpolationMode.BILINEAR)
        else:
            img = TF.resize(img, [336, 336], interpolation=TF.InterpolationMode.BILINEAR)
            img = TF.center_crop(img, [self.image_size, self.image_size])

        x = TF.normalize(TF.to_tensor(img), mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        y = torch.tensor([float(row[c]) for c in CLASSES], dtype=torch.float32)
        return x, y


class ClassifierNet(nn.Module):
    def __init__(self, num_classes: int = 20):
        super().__init__()
        try:
            backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        except Exception as e:
            print(f'[CLS] Pretrained ResNet50 unavailable ({e}); using random init.', flush=True)
            backbone = models.resnet50(weights=None)
        self.backbone = nn.Sequential(*list(backbone.children())[:-2])
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.head = nn.Sequential(
            nn.Linear(2048, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.35),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.25),
            nn.Linear(256, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.pool(self.backbone(x)).flatten(1))


def f1_image(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    tp = float(np.logical_and(y_true == 1, y_pred == 1).sum())
    fp = float(np.logical_and(y_true == 0, y_pred == 1).sum())
    fn = float(np.logical_and(y_true == 1, y_pred == 0).sum())
    if tp == 0.0 and fp == 0.0 and fn == 0.0:
        return 1.0
    return (2.0 * tp) / (2.0 * tp + fp + fn)


def focal_loss_fn(logits: torch.Tensor, targets: torch.Tensor, alpha: float = 0.25, gamma: float = 2.0) -> torch.Tensor:
    p = torch.sigmoid(logits)
    ce = -alpha * targets * torch.log(p + 1e-8) - (1 - alpha) * (1 - targets) * torch.log(1 - p + 1e-8)
    pt = torch.where(targets == 1, p, 1 - p)
    return ((1 - pt).pow(gamma) * ce).mean()


def _class_threshold_search(probs: np.ndarray, targets: np.ndarray, grid: np.ndarray) -> Dict[str, float]:
    thresholds: Dict[str, float] = {}
    for class_idx, class_name in enumerate(CLASSES):
        best_thr = 0.5
        best_f1 = -1.0
        y_true = targets[:, class_idx]
        y_prob = probs[:, class_idx]
        for thr in grid:
            y_pred = (y_prob >= float(thr)).astype(np.int64)
            tp = float(np.logical_and(y_true == 1, y_pred == 1).sum())
            fp = float(np.logical_and(y_true == 0, y_pred == 1).sum())
            fn = float(np.logical_and(y_true == 1, y_pred == 0).sum())
            if tp == 0.0 and fp == 0.0 and fn == 0.0:
                score = 1.0
            else:
                score = (2.0 * tp) / (2.0 * tp + fp + fn)
            if score > best_f1:
                best_f1 = score
                best_thr = float(thr)
        thresholds[class_name] = round(best_thr, 4)
    return thresholds


def _mean_image_f1(probs: np.ndarray, targets: np.ndarray, thresholds: Dict[str, float]) -> float:
    thr_vec = np.array([thresholds[c] for c in CLASSES], dtype=np.float32)
    preds = (probs >= thr_vec[None, :]).astype(np.int64)
    scores = [f1_image(targets[i], preds[i]) for i in range(len(targets))]
    return float(np.mean(scores)) if scores else 0.0


def train_classifier_from_scratch(
    epochs: int = 52,
    batch_size: int = 16,
    lr: float = 2.5e-4,
    val_ratio: float = 0.05,
    resume: bool = False,
    use_amp: bool = True,
) -> Tuple[Path, pd.DataFrame, Dict[str, float]]:
    labels_path = DATA_ROOT / 'train' / 'labels.csv'
    img_dir = DATA_ROOT / 'train' / 'images'
    df = pd.read_csv(labels_path).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

    split = int((1.0 - val_ratio) * len(df))
    tr_df = df.iloc[:split].copy()
    va_df = df.iloc[split:].copy()

    y_train = tr_df[CLASSES].to_numpy(dtype=np.float32)
    pos = y_train.sum(axis=0)
    neg = len(y_train) - pos
    pos_weight = torch.tensor((neg + 2.0) / (pos + 1.0), dtype=torch.float32, device=device).clamp(1.0, 15.0)

    tr_loader = DataLoader(
        ClassificationTrainDataset(tr_df, img_dir, train=True, image_size=320),
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=(device.type == 'cuda'),
    )
    va_loader = DataLoader(
        ClassificationTrainDataset(va_df, img_dir, train=False, image_size=320),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=(device.type == 'cuda'),
    )

    model = ClassifierNet(num_classes=len(CLASSES)).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, epochs))
    bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    scaler = torch.amp.GradScaler('cuda', enabled=(use_amp and device.type == 'cuda'))

    out_path = CLASS_WEIGHTS / 'best_classifier.pth'
    threshold_path = CLASS_WEIGHTS / 'class_thresholds.json'
    best_f1 = -1.0
    best_thresholds = {c: 0.5 for c in CLASSES}
    thr_grid = np.arange(0.15, 0.86, 0.02, dtype=np.float32)

    if resume and out_path.exists():
        ckpt = torch.load(out_path, map_location='cpu')
        if isinstance(ckpt, dict) and 'state_dict' in ckpt:
            model.load_state_dict(ckpt['state_dict'], strict=False)
            best_f1 = float(ckpt.get('val_f1', -1.0))
            best_thresholds.update(ckpt.get('best_thresholds', {}))

    for ep in range(1, epochs + 1):
        model.train()
        loss_values = []
        for x, y in tr_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type='cuda', enabled=(use_amp and device.type == 'cuda')):
                logits = model(x)
                loss = 0.75 * bce(logits, y) + 0.25 * focal_loss_fn(logits, y)
            if not torch.isfinite(loss):
                continue
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            loss_values.append(float(loss.item()))

        model.eval()
        val_probs = []
        val_targets = []
        with torch.no_grad():
            for x, y in va_loader:
                x = x.to(device, non_blocking=True)
                logits = model(x)
                probs = torch.sigmoid(logits).cpu().numpy()
                val_probs.append(probs)
                val_targets.append(y.numpy().astype(np.int64))
        val_probs = np.concatenate(val_probs, axis=0) if val_probs else np.zeros((0, len(CLASSES)), dtype=np.float32)
        val_targets = np.concatenate(val_targets, axis=0) if val_targets else np.zeros((0, len(CLASSES)), dtype=np.int64)

        epoch_thresholds = _class_threshold_search(val_probs, val_targets, thr_grid)
        epoch_f1 = _mean_image_f1(val_probs, val_targets, epoch_thresholds)
        scheduler.step()

        print(f'[CLS] Epoch {ep:02d}/{epochs} | train_loss={float(np.mean(loss_values)) if loss_values else 0.0:.4f} | val_f1={epoch_f1:.4f}')

        if epoch_f1 > best_f1:
            best_f1 = epoch_f1
            best_thresholds = epoch_thresholds
            torch.save(
                {
                    'state_dict': model.state_dict(),
                    'val_f1': best_f1,
                    'best_thresholds': best_thresholds,
                },
                out_path,
            )
            with open(threshold_path, 'w', encoding='utf-8') as f:
                json.dump({'__class_thresholds__': best_thresholds}, f, indent=2)
            print(f'[CLS] Saved best checkpoint -> {out_path} (val_f1={best_f1:.4f})')

    return out_path, va_df, best_thresholds


print('[CLS] Starting classifier training...')
classifier_ckpt, val_df_cls, best_thresholds = train_classifier_from_scratch(
    epochs=52,
    batch_size=16,
    lr=2.5e-4,
    val_ratio=0.05,
    resume=False,
    use_amp=True,
    # epochs=36, batch_size=16, lr=2.5e-4, val_ratio=0.10, resume=False, use_amp=True
)
print('[CLS] Best thresholds:', best_thresholds)

In [ ]:
print('[CLS] Training-time threshold tuning is enabled for model selection.')
print('[CLS] Inference/submission uses fixed threshold = 0.5 as required by assignment instructions.')

## 3) Train Segmentation Models A and B

In [ ]:
from torchvision import models

print('[SEG] Using a custom ResNet50 backbone plus a handwritten decoder.')

In [ ]:
class SegmentationDataset(Dataset):
    def __init__(self, image_paths: List[Path], mask_paths: List[Path], image_size: int = 384, augment: bool = False):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.image_size = image_size
        self.augment = augment

    def __len__(self) -> int:
        return len(self.image_paths)

    def __getitem__(self, idx: int):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        mask = Image.open(self.mask_paths[idx])

        if self.augment:
            base = self.image_size + 64
            img = TF.resize(img, [base, base], interpolation=TF.InterpolationMode.BILINEAR)
            mask = TF.resize(mask, [base, base], interpolation=TF.InterpolationMode.NEAREST)
            i, j, h, w = transforms.RandomCrop.get_params(img, output_size=(self.image_size, self.image_size))
            img = TF.crop(img, i, j, h, w)
            mask = TF.crop(mask, i, j, h, w)
            if random.random() < 0.5:
                img = TF.hflip(img)
                mask = TF.hflip(mask)
            if random.random() < 0.25:
                img = TF.vflip(img)
                mask = TF.vflip(mask)
            if random.random() < 0.35:
                img = TF.adjust_brightness(img, 0.75 + 0.5 * random.random())
            if random.random() < 0.35:
                img = TF.adjust_contrast(img, 0.75 + 0.5 * random.random())
            if random.random() < 0.20:
                img = TF.adjust_saturation(img, 0.75 + 0.5 * random.random())
        else:
            img = TF.resize(img, [self.image_size, self.image_size], interpolation=TF.InterpolationMode.BILINEAR)
            mask = TF.resize(mask, [self.image_size, self.image_size], interpolation=TF.InterpolationMode.NEAREST)

        x = TF.normalize(TF.to_tensor(img), mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        y = torch.from_numpy(np.asarray(mask, dtype=np.uint8).astype(np.int64))
        return x, y


class ConvBNAct(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int = 3, stride: int = 1, padding: int | None = None):
        super().__init__()
        if padding is None:
            padding = kernel_size // 2
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride, padding=padding, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class ResNetBackbone(nn.Module):
    def __init__(self, pretrained_backbone: bool = True):
        super().__init__()
        weights = models.ResNet50_Weights.IMAGENET1K_V2 if pretrained_backbone else None
        backbone = models.resnet50(weights=weights)
        self.stem = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
        )
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4

    def forward(self, x: torch.Tensor):
        x = self.stem(x)
        c1 = self.layer1(x)
        c2 = self.layer2(c1)
        c3 = self.layer3(c2)
        c4 = self.layer4(c3)
        return c1, c2, c3, c4


class SegmentationDecoder(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.proj4 = ConvBNAct(2048, 256, kernel_size=1, padding=0)
        self.fuse34 = nn.Sequential(
            ConvBNAct(256 + 1024, 256),
            ConvBNAct(256, 256),
        )
        self.fuse23 = nn.Sequential(
            ConvBNAct(256 + 512, 192),
            ConvBNAct(192, 192),
        )
        self.fuse12 = nn.Sequential(
            ConvBNAct(192 + 256, 128),
            ConvBNAct(128, 128),
        )
        self.classifier = nn.Conv2d(128, num_classes, kernel_size=1)

    def forward(self, c1: torch.Tensor, c2: torch.Tensor, c3: torch.Tensor, c4: torch.Tensor, out_size: Tuple[int, int]) -> torch.Tensor:
        x = self.proj4(c4)
        x = F.interpolate(x, size=c3.shape[-2:], mode='bilinear', align_corners=False)
        x = self.fuse34(torch.cat([x, c3], dim=1))
        x = F.interpolate(x, size=c2.shape[-2:], mode='bilinear', align_corners=False)
        x = self.fuse23(torch.cat([x, c2], dim=1))
        x = F.interpolate(x, size=c1.shape[-2:], mode='bilinear', align_corners=False)
        x = self.fuse12(torch.cat([x, c1], dim=1))
        x = F.interpolate(x, size=out_size, mode='bilinear', align_corners=False)
        return self.classifier(x)


class CustomSegmentationNet(nn.Module):
    def __init__(self, num_classes: int = NUM_CLASSES, pretrained_backbone: bool = True):
        super().__init__()
        self.backbone = ResNetBackbone(pretrained_backbone=pretrained_backbone)
        self.decoder = SegmentationDecoder(num_classes=num_classes)

    def forward(self, x: torch.Tensor):
        c1, c2, c3, c4 = self.backbone(x)
        logits = self.decoder(c1, c2, c3, c4, x.shape[-2:])
        return {'out': logits}


def split_seg_data(val_ratio: float = 0.10, seed: int = 42):
    img_dir = DATA_ROOT / 'train' / 'images'
    mask_dir = DATA_ROOT / 'train' / 'segmentation_masks'
    img_paths = sorted(img_dir.glob('*.jpg'))
    pairs = []
    for p in img_paths:
        mp = mask_dir / f'{p.stem}.png'
        if mp.exists():
            pairs.append((p, mp))

    random.Random(seed).shuffle(pairs)
    n_val = max(1, int(len(pairs) * val_ratio))
    va = pairs[:n_val]
    tr = pairs[n_val:]
    tr_i = [a for a, _ in tr]
    tr_m = [b for _, b in tr]
    va_i = [a for a, _ in va]
    va_m = [b for _, b in va]
    return (tr_i, tr_m), (va_i, va_m)


def estimate_class_weights(mask_paths: List[Path], num_classes: int, sample_limit: int = 700) -> torch.Tensor:
    hist = np.zeros(num_classes, dtype=np.float64)
    for p in mask_paths[: min(len(mask_paths), sample_limit)]:
        m = np.asarray(Image.open(p), dtype=np.uint8)
        valid = m[m != 255]
        if valid.size:
            hist += np.bincount(valid, minlength=num_classes).astype(np.float64)
    hist[0] = max(hist[0], 1.0)
    freq = hist / max(1.0, hist.sum())
    w = 1.0 / np.sqrt(np.clip(freq, 1e-8, None))
    w[0] = min(w[0], np.percentile(w[1:], 25))
    w = w / np.mean(w[1:])
    return torch.tensor(np.clip(w, 0.25, 8.0), dtype=torch.float32, device=device)


def soft_dice_loss(logits: torch.Tensor, target: torch.Tensor, ignore_idx: int = 255) -> torch.Tensor:
    probs = torch.softmax(logits, dim=1)
    valid = (target != ignore_idx).unsqueeze(1)
    target = target.clone()
    target[target == ignore_idx] = 0
    onehot = F.one_hot(target, num_classes=NUM_CLASSES).permute(0, 3, 1, 2).float()
    probs = probs * valid
    onehot = onehot * valid
    probs = probs[:, 1:]
    onehot = onehot[:, 1:]
    inter = (probs * onehot).sum(dim=(0, 2, 3))
    denom = probs.sum(dim=(0, 2, 3)) + onehot.sum(dim=(0, 2, 3))
    return 1.0 - ((2.0 * inter + 1.0) / (denom + 1.0)).mean()


def compute_miou(pred: torch.Tensor, target: torch.Tensor, ignore_idx: int = 255) -> float:
    pred = pred.detach().cpu().numpy()
    target = target.detach().cpu().numpy()
    valid = target != ignore_idx
    scores = []
    for c in range(1, NUM_CLASSES):
        p = (pred == c) & valid
        g = (target == c) & valid
        union = np.logical_or(p, g).sum()
        if union > 0:
            inter = np.logical_and(p, g).sum()
            scores.append(float(inter) / float(union))
    return float(np.mean(scores)) if scores else 0.0


def train_segmentation_variant(
    variant_name: str,
    epochs: int = 12,
    batch_size: int = 4,
    image_size: int = 384,
    lr: float = 1.5e-4,
    val_ratio: float = 0.10,
    use_amp: bool = True,
    seed_offset: int = 0,
    dice_weight: float = 0.6,
    label_smoothing: float = 0.02,
    output_name: str | None = None,
) -> Tuple[Path, float]:
    local_seed = SEED + int(seed_offset)
    random.seed(local_seed)
    np.random.seed(local_seed)
    torch.manual_seed(local_seed)

    split_seed = local_seed + 97
    (tr_i, tr_m), (va_i, va_m) = split_seg_data(val_ratio=val_ratio, seed=split_seed)
    tr_loader = DataLoader(
        SegmentationDataset(tr_i, tr_m, image_size=image_size, augment=True),
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=(device.type == 'cuda'),
    )
    va_loader = DataLoader(
        SegmentationDataset(va_i, va_m, image_size=image_size, augment=False),
        batch_size=max(1, batch_size // 2),
        shuffle=False,
        num_workers=0,
        pin_memory=(device.type == 'cuda'),
    )

    model = CustomSegmentationNet(num_classes=NUM_CLASSES, pretrained_backbone=True).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, epochs))
    class_weights = estimate_class_weights(tr_m, NUM_CLASSES, sample_limit=700)
    ce = nn.CrossEntropyLoss(ignore_index=255, weight=class_weights, label_smoothing=label_smoothing)
    scaler = torch.amp.GradScaler('cuda', enabled=(use_amp and device.type == 'cuda'))

    if output_name is None:
        output_name = f'best_segmentation_{variant_name.lower()}.pth'
    out_path = SEG_WEIGHTS / output_name
    best_miou = -1.0

    for ep in range(1, epochs + 1):
        model.train()
        tr_losses = []
        for x, y in tr_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=(use_amp and device.type == 'cuda')):
                logits = model(x)['out']
                loss = ce(logits, y) + dice_weight * soft_dice_loss(logits, y)
            if not torch.isfinite(loss):
                continue
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            tr_losses.append(float(loss.item()))

        model.eval()
        va_losses = []
        va_mious = []
        with torch.no_grad():
            for x, y in va_loader:
                x = x.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)
                with torch.amp.autocast('cuda', enabled=(use_amp and device.type == 'cuda')):
                    logits = model(x)['out']
                    logits_flip = torch.flip(model(torch.flip(x, dims=[3]))['out'], dims=[3])
                    logits = 0.5 * (logits + logits_flip)
                    loss = ce(logits, y) + dice_weight * soft_dice_loss(logits, y)
                va_losses.append(float(loss.item()))
                va_mious.append(compute_miou(torch.argmax(logits, dim=1), y))

        tr_loss = float(np.mean(tr_losses)) if tr_losses else 0.0
        va_loss = float(np.mean(va_losses)) if va_losses else 0.0
        va_miou = float(np.mean(va_mious)) if va_mious else 0.0
        scheduler.step()
        print(f'[SEG-{variant_name}] Epoch {ep:02d}/{epochs} | train_loss={tr_loss:.4f} | val_loss={va_loss:.4f} | val_mIoU={va_miou:.4f}')

        if va_miou > best_miou:
            best_miou = va_miou
            torch.save({'model_state': model.state_dict(), 'val_miou': best_miou, 'variant': variant_name}, out_path)
            print(f'[SEG-{variant_name}] Saved best checkpoint -> {out_path} (mIoU={best_miou:.4f})')

    return out_path, float(best_miou)


print('[SEG] Training two separate segmentation variants (A and B)...')
# Kaggle-quality run: higher resolution, AMP, and distinct hyperparameters

seg_ckpt_a, seg_val_a = train_segmentation_variant(
    variant_name='A',
    epochs=32,
    batch_size=4,
    image_size=384,
    lr=1.5e-4,
    val_ratio=0.10,
    use_amp=True,
    seed_offset=0,
    dice_weight=0.60,
    label_smoothing=0.02,
    output_name='best_segmentation_a.pth',
)

seg_ckpt_b, seg_val_b = train_segmentation_variant(
    variant_name='B',
    epochs=32,
    batch_size=4,
    image_size=384,
    lr=2.0e-4,
    val_ratio=0.10,
    use_amp=True,
    seed_offset=29,
    dice_weight=0.50,
    label_smoothing=0.01,
    output_name='best_segmentation_b.pth',
)

print('[SEG] Variant A checkpoint:', seg_ckpt_a, '| best val mIoU:', seg_val_a)
print('[SEG] Variant B checkpoint:', seg_ckpt_b, '| best val mIoU:', seg_val_b)

In [ ]:
import shutil

def evaluate_segmentation_checkpoint(
    ckpt_path: Path,
    image_size: int = 384,
    val_ratio: float = 0.10,
    seed: int = 42,
    batch_size: int = 2,
) -> Tuple[float, np.ndarray]:
    if not ckpt_path.exists():
        raise FileNotFoundError(f'Missing checkpoint: {ckpt_path}')

    model = CustomSegmentationNet(num_classes=NUM_CLASSES, pretrained_backbone=False).to(device)
    ckpt = torch.load(ckpt_path, map_location='cpu')
    state = ckpt.get('model_state', ckpt) if isinstance(ckpt, dict) else ckpt
    model.load_state_dict(state, strict=True)
    model.eval()

    (_, _), (va_i, va_m) = split_seg_data(val_ratio=val_ratio, seed=seed)
    va_loader = DataLoader(
        SegmentationDataset(va_i, va_m, image_size=image_size, augment=False),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=(device.type == 'cuda'),
    )

    per_image_scores: List[float] = []
    with torch.no_grad():
        for x, y in va_loader:
            x = x.to(device, non_blocking=True)
            y = y.to(device, non_blocking=True)
            logits = model(x)['out']
            logits_flip = torch.flip(model(torch.flip(x, dims=[3]))['out'], dims=[3])
            logits = 0.5 * (logits + logits_flip)
            pred = torch.argmax(logits, dim=1)
            for b in range(pred.shape[0]):
                per_image_scores.append(compute_miou(pred[b], y[b]))

    arr = np.asarray(per_image_scores, dtype=np.float64)
    return float(np.mean(arr)) if arr.size else 0.0, arr


seg_ckpt_a = SEG_WEIGHTS / 'best_segmentation_a.pth'
seg_ckpt_b = SEG_WEIGHTS / 'best_segmentation_b.pth'

score_a, scores_a = evaluate_segmentation_checkpoint(seg_ckpt_a, image_size=384, val_ratio=0.10, seed=SEED + 123, batch_size=2)
score_b, scores_b = evaluate_segmentation_checkpoint(seg_ckpt_b, image_size=384, val_ratio=0.10, seed=SEED + 123, batch_size=2)

print(f'[SEG] Variant A validation mIoU: {score_a:.4f}')
print(f'[SEG] Variant B validation mIoU: {score_b:.4f}')

best_ckpt = seg_ckpt_a if score_a >= score_b else seg_ckpt_b
best_alias = SEG_WEIGHTS / 'best_segmentation.pth'
shutil.copy2(best_ckpt, best_alias)
print(f'[SEG] Selected best checkpoint for submission alias: {best_ckpt} -> {best_alias}')

comparison_df = pd.DataFrame([
    {'variant': 'A', 'checkpoint': str(seg_ckpt_a), 'val_miou': score_a},
    {'variant': 'B', 'checkpoint': str(seg_ckpt_b), 'val_miou': score_b},
])
comparison_csv = SEG_WEIGHTS / 'segmentation_variant_comparison.csv'
comparison_df.to_csv(comparison_csv, index=False)
np.savez(
    SEG_WEIGHTS / 'segmentation_variant_val_scores.npz',
    scores_a=scores_a,
    scores_b=scores_b,
    val_seed=SEED + 123,
    val_ratio=0.10,
    image_size=384,
 )

print('[SEG] Saved variant comparison table:', comparison_csv)
display(comparison_df)

In [ ]:
# Ensure submission-time segmentation model code matches the trained checkpoint architecture.
# Run this cell once before the submission generation cell.

fallback_segmentation_model_code = '''
from __future__ import annotations

from pathlib import Path
from typing import List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torchvision import models, transforms

NUM_CLASSES = 21

class ConvBNAct(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int = 3, stride: int = 1, padding: int | None = None):
        super().__init__()
        if padding is None:
            padding = kernel_size // 2
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride, padding=padding, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class SegmentationBackbone(nn.Module):
    def __init__(self, pretrained_backbone: bool = False):
        super().__init__()
        weights = models.ResNet50_Weights.IMAGENET1K_V2 if pretrained_backbone else None
        backbone = models.resnet50(weights=weights)
        self.stem = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
        )
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4

    def forward(self, x: torch.Tensor):
        x = self.stem(x)
        c1 = self.layer1(x)
        c2 = self.layer2(c1)
        c3 = self.layer3(c2)
        c4 = self.layer4(c3)
        return c1, c2, c3, c4


class SegmentationDecoder(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.proj4 = ConvBNAct(2048, 256, kernel_size=1, padding=0)
        self.fuse34 = nn.Sequential(
            ConvBNAct(256 + 1024, 256),
            ConvBNAct(256, 256),
        )
        self.fuse23 = nn.Sequential(
            ConvBNAct(256 + 512, 192),
            ConvBNAct(192, 192),
        )
        self.fuse12 = nn.Sequential(
            ConvBNAct(192 + 256, 128),
            ConvBNAct(128, 128),
        )
        self.classifier = nn.Conv2d(128, num_classes, kernel_size=1)

    def forward(self, c1: torch.Tensor, c2: torch.Tensor, c3: torch.Tensor, c4: torch.Tensor, out_size: Tuple[int, int]) -> torch.Tensor:
        x = self.proj4(c4)
        x = F.interpolate(x, size=c3.shape[-2:], mode='bilinear', align_corners=False)
        x = self.fuse34(torch.cat([x, c3], dim=1))
        x = F.interpolate(x, size=c2.shape[-2:], mode='bilinear', align_corners=False)
        x = self.fuse23(torch.cat([x, c2], dim=1))
        x = F.interpolate(x, size=c1.shape[-2:], mode='bilinear', align_corners=False)
        x = self.fuse12(torch.cat([x, c1], dim=1))
        x = F.interpolate(x, size=out_size, mode='bilinear', align_corners=False)
        return self.classifier(x)


class CustomSegmentationNet(nn.Module):
    def __init__(self, num_classes: int = NUM_CLASSES, pretrained_backbone: bool = False):
        super().__init__()
        self.backbone = SegmentationBackbone(pretrained_backbone=pretrained_backbone)
        self.decoder = SegmentationDecoder(num_classes=num_classes)

    def forward(self, x: torch.Tensor):
        c1, c2, c3, c4 = self.backbone(x)
        logits = self.decoder(c1, c2, c3, c4, x.shape[-2:])
        return {'out': logits}


class SegmentationModel:
    def __init__(self, weights_dir: str):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = CustomSegmentationNet(num_classes=NUM_CLASSES, pretrained_backbone=False).to(self.device)
        ckpt_path = Path(weights_dir) / 'best_segmentation.pth'
        if not ckpt_path.exists():
            raise FileNotFoundError(f'Segmentation weights not found: {ckpt_path}')
        state = torch.load(ckpt_path, map_location='cpu')
        if isinstance(state, dict):
            state = state.get('model_state', state.get('state_dict', state))
        self.model.load_state_dict(state, strict=True)
        self.model.eval()
        self.transform = transforms.Compose([
            transforms.Resize((384, 384)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def predict(self, image: np.ndarray) -> np.ndarray:
        if image.dtype != np.uint8:
            image = image.astype(np.uint8)
        h, w = image.shape[:2]
        pil_img = Image.fromarray(image, mode='RGB')
        x = self.transform(pil_img).unsqueeze(0).to(self.device)
        with torch.no_grad():
            logits = self.model(x)['out']
            logits_flip = torch.flip(self.model(torch.flip(x, dims=[3]))['out'], dims=[3])
            pred = torch.argmax(0.5 * (logits + logits_flip), dim=1).squeeze(0).cpu().numpy().astype(np.uint8)
        if pred.shape != (h, w):
            pred = np.array(Image.fromarray(pred).resize((w, h), Image.NEAREST), dtype=np.uint8, copy=True)
        pred[pred == 255] = 0
        return pred
'''

seg_model_py = SEG_DIR / 'model.py'
source_code = globals().get('segmentation_model_code', fallback_segmentation_model_code)
seg_model_py.write_text(source_code, encoding='utf-8')

ckpt_path = SEG_WEIGHTS / 'best_segmentation.pth'
if not ckpt_path.exists():
    raise FileNotFoundError(f'Segmentation checkpoint missing: {ckpt_path}')

state = torch.load(ckpt_path, map_location='cpu')
if isinstance(state, dict):
    state = state.get('model_state', state.get('state_dict', state))

if not isinstance(state, dict) or len(state) == 0:
    raise RuntimeError('Segmentation checkpoint state_dict is empty or invalid.')

sample_keys = list(state.keys())[:10]
has_custom_keys = any(k.startswith('backbone.') or k.startswith('decoder.') for k in state.keys())
if not has_custom_keys:
    raise RuntimeError(
        'Checkpoint does not look like the custom segmentation model trained in this notebook. '
        f'Sample keys: {sample_keys}'
    )

print('[SEG] Rewrote segmentation/model.py to custom architecture and validated checkpoint format.')
print('[SEG] model.py ->', seg_model_py)
print('[SEG] checkpoint ->', ckpt_path)

In [ ]:
# Kaggle-safe fix: keep existing .pth files, but load only a checkpoint compatible with custom backbone+decoder model.

runtime_safe_segmentation_model_code = '''
from __future__ import annotations

from pathlib import Path
from typing import Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torchvision import models, transforms

NUM_CLASSES = 21

class ConvBNAct(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int = 3, stride: int = 1, padding: int | None = None):
        super().__init__()
        if padding is None:
            padding = kernel_size // 2
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride, padding=padding, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class SegmentationBackbone(nn.Module):
    def __init__(self, pretrained_backbone: bool = False):
        super().__init__()
        weights = models.ResNet50_Weights.IMAGENET1K_V2 if pretrained_backbone else None
        backbone = models.resnet50(weights=weights)
        self.stem = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
        )
        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4

    def forward(self, x: torch.Tensor):
        x = self.stem(x)
        c1 = self.layer1(x)
        c2 = self.layer2(c1)
        c3 = self.layer3(c2)
        c4 = self.layer4(c3)
        return c1, c2, c3, c4


class SegmentationDecoder(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.proj4 = ConvBNAct(2048, 256, kernel_size=1, padding=0)
        self.fuse34 = nn.Sequential(
            ConvBNAct(256 + 1024, 256),
            ConvBNAct(256, 256),
        )
        self.fuse23 = nn.Sequential(
            ConvBNAct(256 + 512, 192),
            ConvBNAct(192, 192),
        )
        self.fuse12 = nn.Sequential(
            ConvBNAct(192 + 256, 128),
            ConvBNAct(128, 128),
        )
        self.classifier = nn.Conv2d(128, num_classes, kernel_size=1)

    def forward(self, c1: torch.Tensor, c2: torch.Tensor, c3: torch.Tensor, c4: torch.Tensor, out_size: Tuple[int, int]) -> torch.Tensor:
        x = self.proj4(c4)
        x = F.interpolate(x, size=c3.shape[-2:], mode='bilinear', align_corners=False)
        x = self.fuse34(torch.cat([x, c3], dim=1))
        x = F.interpolate(x, size=c2.shape[-2:], mode='bilinear', align_corners=False)
        x = self.fuse23(torch.cat([x, c2], dim=1))
        x = F.interpolate(x, size=c1.shape[-2:], mode='bilinear', align_corners=False)
        x = self.fuse12(torch.cat([x, c1], dim=1))
        x = F.interpolate(x, size=out_size, mode='bilinear', align_corners=False)
        return self.classifier(x)


class CustomSegmentationNet(nn.Module):
    def __init__(self, num_classes: int = NUM_CLASSES, pretrained_backbone: bool = False):
        super().__init__()
        self.backbone = SegmentationBackbone(pretrained_backbone=pretrained_backbone)
        self.decoder = SegmentationDecoder(num_classes=num_classes)

    def forward(self, x: torch.Tensor):
        c1, c2, c3, c4 = self.backbone(x)
        logits = self.decoder(c1, c2, c3, c4, x.shape[-2:])
        return {'out': logits}


def _extract_state_dict(state):
    if isinstance(state, dict):
        if 'model_state' in state and isinstance(state['model_state'], dict):
            return state['model_state']
        if 'state_dict' in state and isinstance(state['state_dict'], dict):
            return state['state_dict']
    return state


def _is_custom_seg_state_dict(state_dict: dict) -> bool:
    if not isinstance(state_dict, dict) or len(state_dict) == 0:
        return False
    keys = state_dict.keys()
    return any(k.startswith('backbone.') or k.startswith('decoder.') for k in keys)


class SegmentationModel:
    def __init__(self, weights_dir: str):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model = CustomSegmentationNet(num_classes=NUM_CLASSES, pretrained_backbone=False).to(self.device)

        wdir = Path(weights_dir)
        if not wdir.exists():
            raise FileNotFoundError(f'Segmentation weights directory not found: {wdir}')

        candidates = [wdir / 'best_segmentation.pth']
        candidates.extend(sorted(wdir.glob('*.pth')))
        unique_candidates = []
        seen = set()
        for c in candidates:
            key = str(c.resolve()) if c.exists() else str(c)
            if key in seen:
                continue
            seen.add(key)
            unique_candidates.append(c)

        last_error = None
        loaded_path = None
        for ckpt_path in unique_candidates:
            if not ckpt_path.exists():
                continue
            try:
                raw = torch.load(ckpt_path, map_location='cpu')
                state = _extract_state_dict(raw)
                if not _is_custom_seg_state_dict(state):
                    continue
                self.model.load_state_dict(state, strict=True)
                loaded_path = ckpt_path
                break
            except Exception as e:
                last_error = e
                continue

        if loaded_path is None:
            names = [p.name for p in sorted(wdir.glob('*.pth'))]
            raise RuntimeError(
                'Could not find any compatible custom segmentation checkpoint in weights directory. '
                f'Found files: {names}. Last error: {last_error}'
            )

        print(f'[SEG] Loaded checkpoint: {loaded_path}')
        self.model.eval()
        self.transform = transforms.Compose([
            transforms.Resize((384, 384)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def predict(self, image: np.ndarray) -> np.ndarray:
        if image.dtype != np.uint8:
            image = image.astype(np.uint8)
        h, w = image.shape[:2]
        pil_img = Image.fromarray(image, mode='RGB')
        x = self.transform(pil_img).unsqueeze(0).to(self.device)
        with torch.no_grad():
            logits = self.model(x)['out']
            logits_flip = torch.flip(self.model(torch.flip(x, dims=[3]))['out'], dims=[3])
            pred = torch.argmax(0.5 * (logits + logits_flip), dim=1).squeeze(0).cpu().numpy().astype(np.uint8)
        if pred.shape != (h, w):
            pred = np.array(Image.fromarray(pred).resize((w, h), Image.NEAREST), dtype=np.uint8, copy=True)
        pred[pred == 255] = 0
        return pred
'''

seg_model_py = SEG_DIR / 'model.py'
seg_model_py.write_text(runtime_safe_segmentation_model_code, encoding='utf-8')

all_ckpts = sorted((SEG_WEIGHTS).glob('*.pth'))
print('[SEG] Runtime-safe segmentation model.py written.')
print('[SEG] model.py ->', seg_model_py)
print('[SEG] Available checkpoint files:', [p.name for p in all_ckpts])

## 4) Generate `submission.csv`

In [ ]:
import importlib.util
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
from PIL import Image


def load_module(module_name: str, file_path: Path):
    spec = importlib.util.spec_from_file_location(module_name, str(file_path))
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module


def mask_to_rle(mask: np.ndarray) -> str:
    flat = mask.reshape(-1)
    parts: List[str] = []
    i = 0
    n = len(flat)
    while i < n:
        value = int(flat[i])
        if value == 0:
            i += 1
            continue
        start = i
        i += 1
        while i < n and int(flat[i]) == value:
            i += 1
        parts.extend([str(start), str(i - start), str(value)])
    return ' '.join(parts) if parts else '0 0 0'


def classify_labels(probs: Dict[str, float], threshold: float = 0.5) -> str:
    labels = [c for c in CLASSES if float(probs.get(c, 0.0)) >= float(threshold)]
    return ' '.join(labels) if labels else 'background'


cls_mod = load_module('cls_model_mod', CLASS_DIR / 'model.py')
seg_mod = load_module('seg_model_mod', SEG_DIR / 'model.py')
cls_model = cls_mod.ClassificationModel(str(CLASS_WEIGHTS))
seg_model = seg_mod.SegmentationModel(str(SEG_WEIGHTS))

fixed_threshold = 0.5
print(f'[SUB] Using fixed classification threshold: {fixed_threshold:.1f}')

test_dir = DATA_ROOT / 'test' / 'images'
image_paths = sorted(test_dir.glob('*.jpg'))
if not image_paths:
    image_paths = sorted(test_dir.glob('*.png'))

rows = []
for idx, path in enumerate(image_paths, start=1):
    image = np.asarray(Image.open(path).convert('RGB'), dtype=np.uint8)
    probs = cls_model.predict(image)
    classification = classify_labels(probs, threshold=fixed_threshold)
    mask = seg_model.predict(image)
    rows.append({
        'image_id': path.stem,
        'classification': classification,
        'segmentation_rle': mask_to_rle(mask),
    })
    if idx % 50 == 0 or idx == len(image_paths):
        print(f'Processed {idx}/{len(image_paths)}')

sub_path = SUBMISSION_ROOT / 'submission.csv'
pd.DataFrame(rows, columns=['image_id', 'classification', 'segmentation_rle']).to_csv(sub_path, index=False)
print('Saved submission to:', sub_path)
